# Cadenas · Cifrados y transformaciones

Cifrados clásicos (Polibio, código de golpes, sustitución con palabra clave, pares de intercambio, escítala, transposición por columnas y XOR) y transformaciones de cadenas: Pig Latin, números romanos, contadores alfanuméricos, palabras de Fibonacci, solapamientos, texto en vertical, reducciones con pila, grupos de paréntesis, rótulos que se desplazan, teclados telefónicos, cifras significativas y mayúsculas en títulos. Todo cifrado se comprueba de ida y vuelta, y cada transformación con `assert` contra una segunda solución o la biblioteca estándar. Ideas de [Edabit](https://edabit.com/es/challenges/python3) y de los [ejercicios de José A. Alonso](https://github.com/jaalonso/Ejercicios-Python).

## 1. Cuadrado de Polibio y código de golpes

El cuadrado de Polibio coloca 25 letras en una cuadrícula de 5 × 5 (la `I` y la `J` comparten casilla) y cambia cada letra por su fila y su columna: `A = 11`, `B = 12`… Escribir `polibio_cifra` y `polibio_descifra` sobre una función `cuadricula(alfabeto)` reutilizable. Después, escribir el código de golpes de las cárceles, que usa una cuadrícula sin `K` (se escribe `C`) y transmite cada letra como dos tandas de golpes: `B` es `. ..` (fila 1, columna 2). Las letras se separan con dos espacios. Quitar tildes y signos antes de cifrar y comprobar el viaje de ida y vuelta.

In [1]:
import random
import re
import string
import unicodedata
from collections import Counter, deque
from decimal import Decimal
from functools import reduce
from itertools import cycle, islice, product, zip_longest
from math import prod


def solo_letras(texto):
    """Mayúsculas de la A a la Z, sin tildes ni signos (la Ñ queda como N)."""
    sin_marcas = unicodedata.normalize("NFD", texto.upper())
    return "".join(c for c in sin_marcas if c in string.ascii_uppercase)


def cuadricula(alfabeto):
    assert len(alfabeto) == 25 == len(set(alfabeto))
    return {letra: divmod(i, 5) for i, letra in enumerate(alfabeto)}


POLIBIO = cuadricula("ABCDEFGHIKLMNOPQRSTUVWXYZ")
DESDE_POLIBIO = {pos: letra for letra, pos in POLIBIO.items()}


def polibio_cifra(texto):
    return " ".join(f"{f + 1}{c + 1}" for f, c in (POLIBIO[l] for l in solo_letras(texto).replace("J", "I")))


def polibio_descifra(cifrado):
    return "".join(DESDE_POLIBIO[int(par[0]) - 1, int(par[1]) - 1] for par in cifrado.split())


for fila in "ABCDE FGHIK LMNOP QRSTU VWXYZ".split():
    print(" ".join(fila))
print(polibio_cifra("Teide"))
print(polibio_descifra(polibio_cifra("¡Jandía!")))

A B C D E
F G H I K
L M N O P
Q R S T U
V W X Y Z
44 15 24 14 15
IANDIA


In [2]:
GOLPES = cuadricula("ABCDEFGHIJLMNOPQRSTUVWXYZ")
DESDE_GOLPES = {pos: letra for letra, pos in GOLPES.items()}


def golpes_cifra(texto):
    return "  ".join("." * (f + 1) + " " + "." * (c + 1)
                     for f, c in (GOLPES[l] for l in solo_letras(texto).replace("K", "C")))


def golpes_descifra(cifrado):
    return "".join(DESDE_GOLPES[tuple(len(t) - 1 for t in letra.split(" "))] for letra in cifrado.split("  "))


print(golpes_cifra("Kayak"))
print(golpes_descifra(golpes_cifra("Kayak")))

assert polibio_cifra("Teide") == "44 15 24 14 15"
assert golpes_cifra("b") == ". .."
for frase in ["Mañana subimos al Teide", "Jinámar y Jandía", "Kilómetro cero en Gáldar"]:
    assert polibio_descifra(polibio_cifra(frase)) == solo_letras(frase).replace("J", "I")
    assert golpes_descifra(golpes_cifra(frase)) == solo_letras(frase).replace("K", "C")

. ...  . .  ..... ....  . .  . ...
CAYAC


## 2. Alfabeto de sustitución con palabra clave

Un alfabeto cifrado se construye escribiendo las letras de una palabra clave sin repetir y, a continuación, el resto del alfabeto en orden. Con `TIMANFAYA` queda `TIMANFYBCDEGH…`. Escribir `alfabeto_clave(clave)` y `sustitucion(texto, clave, descifrar=False)`, que cambie cada letra (mayúscula o minúscula) por la de la misma posición en el otro alfabeto con `str.maketrans`, dejando igual lo demás. Comprobar que el alfabeto cifrado es una permutación y el viaje de ida y vuelta con claves y textos aleatorios.

In [3]:
def alfabeto_clave(clave):
    return "".join(dict.fromkeys(solo_letras(clave) + string.ascii_uppercase))


def sustitucion(texto, clave, descifrar=False):
    origen, destino = string.ascii_uppercase, alfabeto_clave(clave)
    if descifrar:
        origen, destino = destino, origen
    return texto.translate(str.maketrans(origen + origen.lower(), destino + destino.lower()))


print(string.ascii_uppercase)
print(alfabeto_clave("Timanfaya"))
cifrado = sustitucion("Quedamos en Yaiza a las nueve.", "Timanfaya")
print(cifrado)
print(sustitucion(cifrado, "Timanfaya", descifrar=True))

rng = random.Random(2026)
for _ in range(200):
    clave = "".join(rng.choices(string.ascii_letters, k=rng.randint(1, 12)))
    texto = "".join(rng.choices(string.ascii_letters + " .,ñá", k=40))
    assert sorted(alfabeto_clave(clave)) == list(string.ascii_uppercase)
    assert sustitucion(sustitucion(texto, clave), clave, descifrar=True) == texto

ABCDEFGHIJKLMNOPQRSTUVWXYZ
TIMANFYBCDEGHJKLOPQRSUVWXZ
Osnathkq nj Xtczt t gtq jsnun.
Quedamos en Yaiza a las nueve.


## 3. Cifrado por pares de intercambio

Algunos cifrados de juego intercambian las letras de unos pares fijos y dejan igual el resto, como el clásico `GA-DE-RY-PO-LU-KI`. Usar la tabla propia `PA-LM-ER-OS-CI-TU` (la `P` se cambia por la `A` y al revés, la `L` por la `M`…), respetando mayúsculas y minúsculas. Validar que la tabla no repite letras y comprobar que el cifrado es su propio inverso: cifrar dos veces devuelve el texto original.

In [4]:
def tabla_pares(clave):
    pares = clave.upper().split("-")
    letras = "".join(pares)
    if any(len(p) != 2 for p in pares) or len(set(letras)) != len(letras):
        raise ValueError(f"tabla no válida: {clave}")
    cambiadas = "".join(p[::-1] for p in pares)
    return str.maketrans(letras + letras.lower(), cambiadas + cambiadas.lower())


PALMEROS = tabla_pares("PA-LM-ER-OS-CI-TU")


def pares_cifra(texto, tabla=PALMEROS):
    return texto.translate(tabla)


print(pares_cifra("Las Palmas"))
print(pares_cifra("Nos vemos en la plaza de Santa Ana a las ocho"))
for mala in ["PA-LM-AE", "PAL-MO"]:
    try:
        tabla_pares(mala)
    except ValueError as e:
        print("Error:", e)

assert pares_cifra("Las Palmas") == "Mpo Apmlpo"
assert all(PALMEROS[PALMEROS.get(k, k)] == k for k in PALMEROS)
rng = random.Random(1)
for _ in range(200):
    texto = "".join(rng.choices(string.ascii_letters + " ,.", k=30))
    assert pares_cifra(pares_cifra(texto)) == texto

Mpo Apmlpo
Nso vrlso rn mp ampzp dr Opnup Pnp p mpo sihs
Error: tabla no válida: PA-LM-AE
Error: tabla no válida: PAL-MO


## 4. Escítala y transposición por columnas

En la transposición por columnas se escribe el mensaje en filas tan anchas como la clave y se leen las columnas en el orden alfabético de las letras de la clave (si una letra se repite, primero la de la izquierda). La escítala espartana es el caso particular en que las columnas se leen en su orden natural: el número de columnas es el número de caras de la vara. Escribir `transpone(texto, clave)` y `destranspone(cifrado, clave)` sin rellenar la última fila (las columnas tienen longitudes distintas) y `escitala(texto, caras)` reutilizándolas. Mostrar la rejilla de un ejemplo y comprobar el viaje de ida y vuelta con claves y textos aleatorios.

In [5]:
def orden_columnas(clave):
    return sorted(range(len(clave)), key=lambda i: (clave[i], i))


def transpone(texto, clave):
    k = len(clave)
    return "".join(texto[i::k] for i in orden_columnas(clave))


def destranspone(cifrado, clave):
    k, n = len(clave), len(cifrado)
    columnas, inicio = [""] * k, 0
    for i in orden_columnas(clave):
        largo = len(range(i, n, k))
        columnas[i] = cifrado[inicio:inicio + largo]
        inicio += largo
    return "".join("".join(fila) for fila in zip_longest(*columnas, fillvalue=""))


def escitala(texto, caras, descifrar=False):
    return (destranspone if descifrar else transpone)(texto, range(caras))


mensaje, clave = "ATACARALAMANECER", "DRAGO"
print(" ".join(clave))
for i in range(0, len(mensaje), len(clave)):
    print(" ".join(mensaje[i:i + len(clave)]))
cifrado = transpone(mensaje, clave)
print("Cifrado:", cifrado)
print("Escítala de 4 caras:", escitala(mensaje, 4))

assert cifrado == "ALEARARCACAMETAN"
assert escitala(mensaje, 4) == "".join(mensaje[i::4] for i in range(4))
rng = random.Random(7)
for _ in range(300):
    texto = "".join(rng.choices(string.ascii_uppercase, k=rng.randint(0, 40)))
    clave = "".join(rng.choices("ABCDE", k=rng.randint(1, 8)))
    assert destranspone(transpone(texto, clave), clave) == texto
    caras = rng.randint(1, 6)
    assert escitala(escitala(texto, caras), caras, descifrar=True) == texto

D R A G O
A T A C A
R A L A M
A N E C E
R
Cifrado: ALEARARCACAMETAN
Escítala de 4 caras: AAAETRMCAAAECLNR


## 5. XOR con clave repetida

Cifrar los bytes de un mensaje haciendo XOR con una clave que se repite tantas veces como haga falta (`itertools.cycle`) y mostrar el resultado en hexadecimal con `bytes.hex()`. Como $(m \oplus k) \oplus k = m$, la misma función descifra tras `bytes.fromhex`. Comprobar el resultado convirtiendo los bytes en un único entero con `int.from_bytes` y ver la debilidad del cifrado: quien conozca el principio del mensaje obtiene la clave.

In [6]:
def xor_clave(datos: bytes, clave: bytes) -> bytes:
    return bytes(b ^ k for b, k in zip(datos, cycle(clave)))


mensaje = "Nos vemos en el muelle de Agaete a las 8. Trae el señuelo.".encode()
clave = b"tabaiba"
cifrado_hex = xor_clave(mensaje, clave).hex()
print(cifrado_hex)
recuperado = xor_clave(bytes.fromhex(cifrado_hex), clave)
print(recuperado.decode())

cifrado = bytes.fromhex(cifrado_hex)
print("Con el texto conocido «Nos vemos»:", xor_clave(cifrado[:9], b"Nos vemos"))

clave_larga = bytes(islice(cycle(clave), len(mensaje)))
como_entero = int.from_bytes(mensaje) ^ int.from_bytes(clave_larga)
assert como_entero.to_bytes(len(mensaje)) == cifrado
assert recuperado == mensaje
rng = random.Random(3)
for _ in range(200):
    datos, k = rng.randbytes(rng.randint(0, 50)), rng.randbytes(rng.randint(1, 8))
    assert xor_clave(bytes.fromhex(xor_clave(datos, k).hex()), k) == datos

3a0e11411f070c1b12420407420418410f140c0e0d114106044923061504160449034118001141514c412013030449070d541207a2d81704180e4c
Nos vemos en el muelle de Agaete a las 8. Trae el señuelo.
Con el texto conocido «Nos vemos»: b'tabaibata'


## 6. Pig Latin

En el Pig Latin, si una palabra empieza por vocal se le añade `way` (`isla → islaway`); si no, su grupo inicial de consonantes pasa al final seguido de `ay` (`trapo → apotray`). La `u` de `qu` va con la `q` (`queso → esoquay`) y la `y` es consonante al principio y vocal en el resto. Las palabras que empezaban con mayúscula la conservan al principio. Escribir `a_pig_latin(texto, guion="")`, que transforme solo las palabras y respete los signos. Sin marca el proceso no tiene vuelta (`trapo` y `rapot` dan `apotray`), así que con `guion="-"` se separa lo movido (`apo-tray`) y las palabras que empiezan por vocal reciben solo `-ay` (con `-way` no se distinguirían de las que empiezan por `w`). Escribir `desde_pig_latin` para ese formato y comprobar la ida y vuelta.

In [7]:
VOCALES = set("aeiouáéíóúü")
PALABRA = re.compile(r"[^\W\d_]+")


def es_vocal(minus, i):
    if minus[i] == "y":
        return i > 0
    if minus[i] == "u" and i > 0 and minus[i - 1] == "q":
        return False
    return minus[i] in VOCALES


def palabra_a_pig(palabra, guion=""):
    minus = palabra.lower()
    i = next((j for j in range(len(minus)) if es_vocal(minus, j)), len(minus))
    sufijo = minus[:i] + "ay" if i or guion else "way"
    resultado = minus[i:] + guion + sufijo
    return resultado.capitalize() if palabra[0].isupper() else resultado


def a_pig_latin(texto, guion=""):
    return PALABRA.sub(lambda m: palabra_a_pig(m.group(), guion), texto)


def desde_pig_latin(texto):
    def deshace(m):
        resto, movido = m.group(1), m.group(2)
        original = (movido + resto).lower()
        return original.capitalize() if m.group()[0].isupper() else original
    return re.sub(r"([^\W\d_]*)-([^\W\d_]*)ay", deshace, texto)


frase = "Queremos ir a Yaiza el sábado, ¿trae el queso?"
print(a_pig_latin(frase))
print(a_pig_latin(frase, guion="-"))
print(desde_pig_latin(a_pig_latin(frase, guion="-")))
print("trapo y rapot sin guion:", a_pig_latin("trapo rapot"))

assert a_pig_latin("isla trapo queso") == "islaway apotray esoquay"
assert a_pig_latin("Yaiza") == "Aizayay"
for texto in [frase, "Los chicharros de Tenerife, ¡qué ricos!", "Whisky y wifi en el Puerto", "pst, ¿oyes?"]:
    assert desde_pig_latin(a_pig_latin(texto, guion="-")) == texto

Eremosquay irway away Aizayay elway ábadosay, ¿aetray elway esoquay?
Eremos-quay ir-ay a-ay Aiza-yay el-ay ábado-say, ¿ae-tray el-ay eso-quay?
Queremos ir a Yaiza el sábado, ¿trae el queso?
trapo y rapot sin guion: apotray apotray


## 7. De romano a entero con validación

Escribir `a_romano(n)` para $1 \le n \le 3999$ y `desde_romano(texto)`, que convierta un número romano en entero y lance `ValueError` si tiene símbolos extraños o está mal formado (`IIII`, `VV`, `IC`, `XM`, `MMMM`…). Una forma sencilla de validar: calcular el valor sumando cada símbolo (o restándolo si el siguiente es mayor) y exigir que `a_romano` de ese valor devuelva exactamente el texto. Comprobar la ida y vuelta de 1 a 3999 y que la validación coincide con una expresión regular en todas las cadenas de hasta 5 símbolos romanos.

In [8]:
VALORES = [(1000, "M"), (900, "CM"), (500, "D"), (400, "CD"), (100, "C"), (90, "XC"),
           (50, "L"), (40, "XL"), (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")]
SIMBOLOS = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}


def a_romano(n):
    if not 1 <= n <= 3999:
        raise ValueError(f"{n} no se puede escribir en romano clásico")
    partes = []
    for valor, simbolo in VALORES:
        veces, n = divmod(n, valor)
        partes.append(simbolo * veces)
    return "".join(partes)


def desde_romano(texto):
    if not texto or set(texto) - SIMBOLOS.keys():
        raise ValueError(f"{texto!r} tiene símbolos no válidos")
    valores = [SIMBOLOS[c] for c in texto]
    total = sum(-v if v < siguiente else v for v, siguiente in zip(valores, valores[1:] + [0]))
    if not 1 <= total <= 3999:
        raise ValueError(f"{texto!r} se sale del rango 1-3999")
    if a_romano(total) != texto:
        raise ValueError(f"{texto!r} está mal formado: {total} se escribe {a_romano(total)}")
    return total


for texto in ["MMXXVI", "XLIX", "IIII", "VV", "IC", "XM", "MMMM", "IIX", "ABC"]:
    try:
        print(f"{texto:>7} = {desde_romano(texto)}")
    except ValueError as e:
        print(f"{texto:>7}   Error: {e}")

 MMXXVI = 2026
   XLIX = 49
   IIII   Error: 'IIII' está mal formado: 4 se escribe IV
     VV   Error: 'VV' está mal formado: 10 se escribe X
     IC   Error: 'IC' está mal formado: 99 se escribe XCIX
     XM   Error: 'XM' está mal formado: 990 se escribe CMXC
   MMMM   Error: 'MMMM' se sale del rango 1-3999
    IIX   Error: 'IIX' está mal formado: 10 se escribe X
    ABC   Error: 'ABC' tiene símbolos no válidos


In [9]:
PATRON = re.compile(r"M{0,3}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})")


def es_valido(texto):
    try:
        desde_romano(texto)
        return True
    except ValueError:
        return False


assert all(desde_romano(a_romano(n)) == n for n in range(1, 4000))
candidatas = ["".join(p) for largo in range(1, 6) for p in product("IVXLCDM", repeat=largo)]
validas = [c for c in candidatas if es_valido(c)]
assert validas == [c for c in candidatas if PATRON.fullmatch(c)]
assert set(validas) == {r for n in range(1, 4000) if len(r := a_romano(n)) <= 5}
print(f"{len(validas)} de {len(candidatas)} cadenas de hasta 5 símbolos son números romanos válidos")

735 de 19607 cadenas de hasta 5 símbolos son números romanos válidos


## 8. Incrementar una cadena como un contador

Escribir `incrementa(cadena)` para cadenas de letras y cifras ASCII que funcione como un cuentakilómetros: se suma uno al último carácter y, si da la vuelta (`9 → 0`, `z → a`, `Z → A`), se acarrea al anterior (`"az9" → "ba0"`). Si el acarreo sale por la izquierda, se añade un carácter del mismo tipo que el primero (`"99" → "100"`, `"zz" → "aaa"`). Comprobar que con cifras equivale a sumar 1 respetando los ceros a la izquierda y que, empezando en `"a"`, recorre la misma sucesión que la numeración biyectiva en base 26 (`a`, …, `z`, `aa`, `ab`…).

In [10]:
VUELTA = {"9": "0", "z": "a", "Z": "A"}
NUEVO = {"9": "1", "z": "a", "Z": "A"}


def incrementa(cadena):
    if not (cadena.isascii() and cadena.isalnum()):
        raise ValueError(f"{cadena!r} no es alfanumérica")
    caracteres = list(cadena)
    for i in reversed(range(len(caracteres))):
        if caracteres[i] not in VUELTA:
            caracteres[i] = chr(ord(caracteres[i]) + 1)
            return "".join(caracteres)
        caracteres[i] = VUELTA[caracteres[i]]
    return NUEVO[cadena[0]] + "".join(caracteres)


def base26_biyectiva(n):
    letras = ""
    while n:
        n, resto = divmod(n - 1, 26)
        letras = chr(ord("a") + resto) + letras
    return letras


for c in ["az9", "lote09", "Zz9", "99", "zz", "A0", "c3po"]:
    print(f"{c:>7} → {incrementa(c)}")
try:
    incrementa("guagua-12")
except ValueError as e:
    print("Error:", e)

assert incrementa("az9") == "ba0" and incrementa("Zz9") == "AAa0"
assert all(incrementa(str(n).zfill(5)) == str(n + 1).zfill(5) for n in range(99_999))
etiqueta = "a"
for n in range(1, 20_000):
    assert etiqueta == base26_biyectiva(n)
    etiqueta = incrementa(etiqueta)

    az9 → ba0
 lote09 → lote10
    Zz9 → AAa0
     99 → 100
     zz → aaa
     A0 → A1
   c3po → c3pp
Error: 'guagua-12' no es alfanumérica


## 9. Palabras de Fibonacci

Las palabras de Fibonacci se forman como la sucesión, pero concatenando: $S_1 = \texttt{a}$, $S_2 = \texttt{ab}$ y $S_n = S_{n-1} S_{n-2}$. Mostrar las primeras y comprobar que la longitud de $S_n$ es el número de Fibonacci $F_{n+1}$ y que tiene $F_n$ letras `a` y $F_{n-1}$ letras `b`. Después, escribir `letra(n, k)`, que devuelva el carácter $k$ de $S_n$ sin construirla (bajando a $S_{n-1}$ o a $S_{n-2}$ según en qué mitad caiga $k$), compararla con la palabra construida y usarla en $S_{100}$, que tiene más de $5 \cdot 10^{20}$ letras.

In [11]:
def palabras_fibonacci(n):
    palabras = ["a", "ab"]
    while len(palabras) < n:
        palabras.append(palabras[-1] + palabras[-2])
    return palabras[:n]


def longitudes(n):
    largos = [0, 1, 2]
    while len(largos) <= n:
        largos.append(largos[-1] + largos[-2])
    return largos


def letra(n, k):
    largos = longitudes(n)
    if not 0 <= k < largos[n]:
        raise IndexError(k)
    while n > 2:
        if k < largos[n - 1]:
            n -= 1
        else:
            k -= largos[n - 1]
            n -= 2
    return "aab"[k + n - 1]


def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a


for n, palabra in enumerate(palabras_fibonacci(7), start=1):
    print(f"S{n} = {palabra}")
print(f"S100 tiene {longitudes(100)[100]:,} letras; la de la posición 10^20 es {letra(100, 10**20)!r}")

palabras = palabras_fibonacci(22)
for n, palabra in enumerate(palabras, start=1):
    assert len(palabra) == fibonacci(n + 1)
    assert Counter(palabra) == Counter(a=fibonacci(n), b=fibonacci(n - 1))
    assert "".join(letra(n, k) for k in range(len(palabra))) == palabra
assert all(palabras[i + 1].startswith(palabras[i]) for i in range(len(palabras) - 1))

S1 = a
S2 = ab
S3 = aba
S4 = abaab
S5 = abaababa
S6 = abaababaabaab
S7 = abaababaabaababaababa
S100 tiene 573,147,844,013,817,084,101 letras; la de la posición 10^20 es 'b'


## 10. Fusionar palabras por su solapamiento

Escribir `solape(a, b)`, que devuelva la longitud del trozo más largo que es a la vez final de `a` y principio de `b`, y `fusiona(a, b)`, que las una sin repetir ese trozo (`garaje` + `jefe` → `garajefe`). Encadenar varias palabras con `functools.reduce`. Comprobar `solape` en cadenas aleatorias contra una segunda versión con la función de prefijos de KMP sobre `b + separador + a`.

In [12]:
def solape(a, b):
    return next((k for k in range(min(len(a), len(b)), 0, -1) if a.endswith(b[:k])), 0)


def solape_kmp(a, b):
    s = b + "\0" + a
    prefijo = [0] * len(s)
    for i in range(1, len(s)):
        j = prefijo[i - 1]
        while j and s[i] != s[j]:
            j = prefijo[j - 1]
        prefijo[i] = j + (s[i] == s[j])
    return prefijo[-1]


def fusiona(a, b):
    return a + b[solape(a, b):]


cadena = ["garaje", "jefe", "fermento", "mentol", "olivo"]
print(" + ".join(cadena), "→", reduce(fusiona, cadena))
for a, b in [("tabaiba", "baifo"), ("cardón", "dónde"), ("roque", "roque"), ("mar", "sol")]:
    print(f"{a} + {b}: solape {solape(a, b)} → {fusiona(a, b)}")

assert reduce(fusiona, cadena) == "garajefermentolivo"
rng = random.Random(5)
for _ in range(3000):
    a = "".join(rng.choices("ab", k=rng.randint(0, 10)))
    b = "".join(rng.choices("ab", k=rng.randint(0, 10)))
    assert solape(a, b) == solape_kmp(a, b)

garaje + jefe + fermento + mentol + olivo → garajefermentolivo
tabaiba + baifo: solape 2 → tabaibaifo
cardón + dónde: solape 3 → cardónde
roque + roque: solape 5 → roque
mar + sol: solape 0 → marsol


## 11. Leer en vertical

Escribir `en_vertical(palabras)`, que coloque cada palabra en una columna para leerlas de arriba abajo: la fila $i$ contiene la letra $i$ de cada palabra, con espacios donde una palabra ya ha terminado (`itertools.zip_longest`) y sin espacios al final de la línea. Escribir `desde_vertical(lineas)`, que recupere las palabras, y comprobar la ida y vuelta con listas aleatorias de palabras no vacías.

In [13]:
def en_vertical(palabras):
    return ["".join(fila).rstrip() for fila in zip_longest(*palabras, fillvalue=" ")]


def desde_vertical(lineas):
    return ["".join(columna).rstrip() for columna in zip_longest(*lineas, fillvalue=" ")]


islas = ["Tenerife", "La Gomera", "El Hierro", "La Palma"]
for linea in en_vertical(islas):
    print(f"|{linea}")
print(desde_vertical(en_vertical(islas)))

assert en_vertical(["ab", "c"]) == ["ac", "b"]
rng = random.Random(11)
for _ in range(500):
    palabras = ["".join(rng.choices("xyz ", k=rng.randint(0, 6))) + rng.choice("xyz") for _ in range(rng.randint(1, 6))]
    assert desde_vertical(en_vertical(palabras)) == palabras

|TLEL
|eala
|n
|eGHP
|roia
|imel
|ferm
|erra
| ao
['Tenerife', 'La Gomera', 'El Hierro', 'La Palma']


## 12. Reducir pares adyacentes con una pila

Eliminar de una cadena cada par de caracteres adyacentes iguales y repetir mientras queden (`"abbaca"`: al quitar `bb` queda `"aaca"` y al quitar `aa`, `"ca"`). Hacerlo en una sola pasada con una pila: cada carácter anula la cima si forma pareja con ella y, si no, se apila. Generalizar la regla de qué es una pareja con una función, para reducir también letras iguales de distinto tamaño (`aA` o `Aa`). Comprobar las dos variantes contra la versión ingenua que aplica `re.sub` hasta que la cadena deja de cambiar.

In [14]:
def reduce_pares(texto, pareja=lambda x, y: x == y):
    pila = []
    for c in texto:
        if pila and pareja(pila[-1], c):
            pila.pop()
        else:
            pila.append(c)
    return "".join(pila)


def reduce_ingenuo(texto, patron=r"(.)\1"):
    while (nuevo := re.sub(patron, "", texto)) != texto:
        texto = nuevo
    return texto


def opuestas(x, y):
    return x != y and x.lower() == y.lower()


OPUESTAS = "|".join(f"{c}{c.upper()}|{c.upper()}{c}" for c in string.ascii_lowercase)

for texto in ["abbaca", "aabccbdeed", "tabaiba"]:
    print(f"{texto!r:14} → {reduce_pares(texto)!r}")
print(f"{'TbBaAtAcabCcBA'!r:14} → {reduce_pares('TbBaAtAcabCcBA', opuestas)!r} (mayúscula y minúscula se anulan)")

rng = random.Random(9)
for _ in range(3000):
    texto = "".join(rng.choices("abc", k=rng.randint(0, 14)))
    assert reduce_pares(texto) == reduce_ingenuo(texto)
    texto = "".join(rng.choices("abAB", k=rng.randint(0, 14)))
    assert reduce_pares(texto, opuestas) == reduce_ingenuo(texto, OPUESTAS)

'abbaca'       → 'ca'
'aabccbdeed'   → ''
'tabaiba'      → 'tabaiba'
'TbBaAtAcabCcBA' → 'Ac' (mayúscula y minúscula se anulan)


## 13. Grupos de paréntesis de primer nivel

Una cadena de paréntesis equilibrada se divide en grupos de primer nivel: los trozos que se cierran cada vez que la profundidad vuelve a cero (`"(()())(())()"` → `["(()())", "(())", "()"]`). Escribir `grupos(texto)`, que lance `ValueError` si la cadena no está equilibrada, y `sin_externos(texto)`, que quite el par exterior de cada grupo. Mostrar la profundidad máxima de cada grupo y comprobar con cadenas equilibradas aleatorias que los grupos vuelven a formar la cadena, que cada uno está equilibrado y que ningún prefijo propio suyo lo está.

In [15]:
def grupos(texto):
    resultado, profundidad, inicio = [], 0, 0
    for i, c in enumerate(texto):
        if c not in "()":
            raise ValueError(f"carácter {c!r} en la posición {i}")
        profundidad += 1 if c == "(" else -1
        if profundidad < 0:
            raise ValueError(f"cierre sin abrir en la posición {i}")
        if profundidad == 0:
            resultado.append(texto[inicio:i + 1])
            inicio = i + 1
    if profundidad:
        raise ValueError(f"quedan {profundidad} paréntesis sin cerrar")
    return resultado


def sin_externos(texto):
    return "".join(g[1:-1] for g in grupos(texto))


def profundidades(texto):
    actual = 0
    for c in texto:
        actual += 1 if c == "(" else -1
        yield actual


texto = "(()())(())()((()))"
for g in grupos(texto):
    print(f"{g:<10} profundidad máxima {max(profundidades(g))}")
print("Sin externos:", sin_externos(texto))
for malo in ["(()", "())(", "(a)"]:
    try:
        grupos(malo)
    except ValueError as e:
        print(f"{malo}: {e}")

(()())     profundidad máxima 2
(())       profundidad máxima 2
()         profundidad máxima 1
((()))     profundidad máxima 3
Sin externos: ()()()(())
((): quedan 1 paréntesis sin cerrar
())(: cierre sin abrir en la posición 2
(a): carácter 'a' en la posición 1


In [16]:
def equilibrada_aleatoria(n, rng):
    letras, abiertos = [], 0
    while n or abiertos:
        if n and (not abiertos or rng.random() < 0.5):
            letras.append("(")
            abiertos, n = abiertos + 1, n - 1
        else:
            letras.append(")")
            abiertos -= 1
    return "".join(letras)


rng = random.Random(4)
for _ in range(1000):
    texto = equilibrada_aleatoria(rng.randint(0, 12), rng)
    partes = grupos(texto)
    assert "".join(partes) == texto
    assert len(partes) == list(profundidades(texto)).count(0)
    for g in partes:
        alturas = list(profundidades(g))
        assert alturas[-1] == 0 and 0 not in alturas[:-1] and min(alturas) >= 0

## 14. ¿Se puede escribir la nota con el recorte?

Para escribir una nota anónima se recortan letras de un periódico. Escribir `puede_escribirse(nota, recorte)`, que diga si el recorte tiene suficientes letras de cada clase (sin distinguir mayúsculas ni tildes, con la `ñ` como letra propia e ignorando espacios y signos), comparando dos `Counter` con `<=`, y `letras_que_faltan(nota, recorte)` con la resta de `Counter`. Comprobarlo contra una versión que va tachando letras de una lista.

In [17]:
SIN_TILDES = str.maketrans("áéíóúü", "aeiouu")


def letras(texto):
    return Counter(c for c in texto.lower().translate(SIN_TILDES) if c.isalpha())


def puede_escribirse(nota, recorte):
    return letras(nota) <= letras(recorte)


def letras_que_faltan(nota, recorte):
    return letras(nota) - letras(recorte)


def puede_escribirse_tachando(nota, recorte):
    disponibles = [c for c in recorte.lower().translate(SIN_TILDES) if c.isalpha()]
    for c in nota.lower().translate(SIN_TILDES):
        if c.isalpha():
            if c not in disponibles:
                return False
            disponibles.remove(c)
    return True


recorte = "El Cabildo aprueba la mejora de la carretera de Teno tras las lluvias de enero"
for nota in ["Deja la bolsa en el Teno", "Trae el dinero mañana", "Cobarde"]:
    faltan = letras_que_faltan(nota, recorte)
    print(f"{nota!r}: {'sí' if puede_escribirse(nota, recorte) else 'no, faltan ' + str(dict(faltan))}")

rng = random.Random(8)
for _ in range(2000):
    nota = "".join(rng.choices("abcñá E", k=rng.randint(0, 8)))
    texto = "".join(rng.choices("abcñaé ,", k=rng.randint(0, 15)))
    assert puede_escribirse(nota, texto) == puede_escribirse_tachando(nota, texto) == (not letras_que_faltan(nota, texto))

'Deja la bolsa en el Teno': sí
'Trae el dinero mañana': no, faltan {'ñ': 1}
'Cobarde': sí


## 15. Un rótulo que se desplaza

Un panel de una estación muestra solo `ancho` caracteres y el texto entra por la derecha y sale por la izquierda. Escribir `fotogramas(texto, ancho)`, que devuelva todos los fotogramas desde el panel vacío hasta que el texto ha salido del todo. Escribir también `fotogramas_circulares(texto, ancho, n)`, para un rótulo que da vueltas sin fin con un separador, de dos formas: por índices módulo la longitud y girando una `deque` con `rotate(-1)`. Comprobar las longitudes, el número de fotogramas y que el rótulo circular se repite con periodo igual a la longitud de la cinta.

In [18]:
def fotogramas(texto, ancho):
    tira = " " * ancho + texto + " " * ancho
    return [tira[i:i + ancho] for i in range(len(texto) + ancho + 1)]


def fotogramas_circulares(texto, ancho, n, separador=" · "):
    cinta = texto + separador
    return ["".join(cinta[(i + j) % len(cinta)] for j in range(ancho)) for i in range(n)]


def fotogramas_deque(texto, ancho, n, separador=" · "):
    cinta, resultado = deque(texto + separador), []
    for _ in range(n):
        resultado.append("".join(islice(cycle(cinta), ancho)))
        cinta.rotate(-1)
    return resultado


for f in fotogramas("Guagua 12", 6):
    print(f"[{f}]")
print()
for f in fotogramas_circulares("Próxima parada: Vegueta", 14, 6):
    print(f"[{f}]")

marcos = fotogramas("Guagua 12", 6)
assert len(marcos) == len("Guagua 12") + 7 and all(len(f) == 6 for f in marcos)
assert marcos[0] == marcos[-1] == " " * 6
assert fotogramas("ab", 1) == [" ", "a", "b", " "]
largo = len("Próxima parada: Vegueta · ")
circulares = fotogramas_circulares("Próxima parada: Vegueta", 30, 3 * largo)
assert circulares == fotogramas_deque("Próxima parada: Vegueta", 30, 3 * largo)
assert all(circulares[i] == circulares[i + largo] for i in range(2 * largo))

[      ]
[     G]
[    Gu]
[   Gua]
[  Guag]
[ Guagu]
[Guagua]
[uagua ]
[agua 1]
[gua 12]
[ua 12 ]
[a 12  ]
[ 12   ]
[12    ]
[2     ]
[      ]

[Próxima parada]
[róxima parada:]
[óxima parada: ]
[xima parada: V]
[ima parada: Ve]
[ma parada: Veg]


## 16. Palabras de un teclado telefónico

En un teclado telefónico, `2 → abc`, `3 → def`, …, `7 → pqrs`, …, `9 → wxyz`. Generar todas las cadenas de letras que corresponden a una secuencia de dígitos de dos formas: con `itertools.product` y con *backtracking*, y comprobar que coinciden (en el mismo orden) y que su número es el producto de las letras de cada tecla. Después, buscar las palabras de un pequeño diccionario que encajan con una secuencia, podando el *backtracking* cuando el prefijo no empieza ninguna palabra, y compararlo con filtrar todas las combinaciones.

In [19]:
TECLAS = {"2": "abc", "3": "def", "4": "ghi", "5": "jkl", "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz"}


def combinaciones(digitos):
    return ["".join(p) for p in product(*(TECLAS[d] for d in digitos))] if digitos else []


def combinaciones_bt(digitos, prefijos=None):
    resultado = []

    def explora(i, prefijo):
        if prefijos is not None and prefijo not in prefijos:
            return
        if i == len(digitos):
            resultado.append(prefijo)
            return
        for letra in TECLAS[digitos[i]]:
            explora(i + 1, prefijo + letra)

    if digitos:
        explora(0, "")
    return resultado


print(combinaciones("27"))
print(len(combinaciones("2779")), "combinaciones para 2779")

['ap', 'aq', 'ar', 'as', 'bp', 'bq', 'br', 'bs', 'cp', 'cq', 'cr', 'cs']
192 combinaciones para 2779


In [20]:
DICCIONARIO = {"casa", "cosa", "boca", "coca", "bola", "cola", "cama", "cara", "mesa", "pala", "sala",
               "rosa", "playa", "mar", "pan", "ola", "nado", "mano"}
PREFIJOS = {p[:i] for p in DICCIONARIO for i in range(len(p) + 1)}
A_DIGITO = {letra: d for d, letras in TECLAS.items() for letra in letras}

for digitos in ["2272", "2622", "2652", "6266", "75292"]:
    encontradas = combinaciones_bt(digitos, PREFIJOS)
    encontradas = [p for p in encontradas if p in DICCIONARIO]
    print(f"{digitos}: {encontradas}")
    assert encontradas == [p for p in combinaciones(digitos) if p in DICCIONARIO]

for p in DICCIONARIO:
    assert p in combinaciones("".join(A_DIGITO[c] for c in p))
rng = random.Random(6)
for _ in range(200):
    digitos = "".join(rng.choices("23456789", k=rng.randint(0, 5)))
    assert combinaciones(digitos) == combinaciones_bt(digitos)
    assert len(combinaciones(digitos)) == (prod(len(TECLAS[d]) for d in digitos) if digitos else 0)

2272: ['cara', 'casa']
2622: ['boca', 'coca']
2652: ['bola', 'cola']
6266: ['mano']
75292: ['playa']


## 17. Cifras significativas

Escribir `cifras_significativas(texto)` para un número escrito como cadena: no cuentan los ceros a la izquierda; sí los que están entre otras cifras; los ceros finales cuentan solo si el número lleva punto decimal (`"1500"` tiene 2, `"1500."` tiene 4 y `"0.00120"` tiene 3); en notación científica solo cuenta la mantisa (`"1.20e3"` tiene 3). El cero y los textos que no son números lanzan `ValueError`. Comprobar los números con punto contra `Decimal(texto).as_tuple().digits` y que formatear un número con `.{n}e` deja exactamente $n$ cifras significativas.

In [21]:
NUMERO = re.compile(r"[+-]?(\d+\.?\d*|\.\d+)([eE][+-]?\d+)?")


def cifras_significativas(texto):
    texto = texto.strip()
    if not NUMERO.fullmatch(texto):
        raise ValueError(f"{texto!r} no es un número")
    mantisa = re.split("[eE]", texto.lstrip("+-"))[0]
    cifras = mantisa.replace(".", "").lstrip("0")
    if not cifras:
        raise ValueError(f"{texto!r} es cero: sus cifras significativas no están bien definidas")
    if "." not in mantisa:
        cifras = cifras.rstrip("0")
    return len(cifras)


for texto in ["1500", "1500.", "0.00120", "1.20e3", "-0.0450", "1001", "007", "6.022E23", "0.000", "1,5"]:
    try:
        print(f"{texto:>8}: {cifras_significativas(texto)}")
    except ValueError as e:
        print(f"{texto:>8}: Error: {e}")

rng = random.Random(12)
for _ in range(2000):
    entera = str(rng.randint(0, 10**rng.randint(0, 5)))
    decimales = "".join(rng.choices("0123456789", k=rng.randint(0, 5)))
    texto = f"{entera}.{decimales}"
    if Decimal(texto) != 0:
        assert cifras_significativas(texto) == len(Decimal(texto).as_tuple().digits)
    n = rng.randint(1, 8)
    x = rng.uniform(1e-6, 1e6)
    assert cifras_significativas(f"{x:.{n - 1}e}") == n
    entero = str(rng.randint(1, 10**6))
    assert cifras_significativas(entero) == len(entero.rstrip("0"))

    1500: 2
   1500.: 4
 0.00120: 3
  1.20e3: 3
 -0.0450: 3
    1001: 4
     007: 1
6.022E23: 4
   0.000: Error: '0.000' es cero: sus cifras significativas no están bien definidas
     1,5: Error: '1,5' no es un número


## 18. Mayúsculas en los títulos

En inglés los títulos suelen escribirse con mayúscula inicial en cada palabra salvo las palabras menores (artículos, conjunciones y preposiciones cortas), que van en minúscula excepto si abren el título. En español, la norma de la RAE es más simple: solo la primera palabra y los nombres propios. Escribir `titulo_ingles(texto)` y `titulo_espanol(texto, propios)`, que reciban el texto con cualquier mezcla de mayúsculas. Comprobar ejemplos, que aplicarlas dos veces no cambia nada y que `titulo_ingles` coincide con `str.title()` cuando no hay palabras menores.

In [22]:
MENORES = {"a", "an", "the", "and", "but", "or", "nor", "for", "so", "yet",
           "of", "in", "on", "at", "to", "by", "up", "as", "off", "per", "via"}


def mayuscula_inicial(palabra):
    return palabra[:1].upper() + palabra[1:]


def titulo_ingles(texto):
    palabras = texto.lower().split()
    return " ".join(p if i and p in MENORES else mayuscula_inicial(p) for i, p in enumerate(palabras))


def titulo_espanol(texto, propios=()):
    por_minusculas = {p.lower(): p for p in propios}
    palabras = [por_minusculas.get(p, p) for p in texto.lower().split()]
    return " ".join([mayuscula_inicial(palabras[0]), *palabras[1:]]) if palabras else ""


ejemplos_en = ["the lord of the rings", "A TALE OF TWO CITIES", "war and peace", "of mice and men"]
for t in ejemplos_en:
    print(f"{t!r:26} → {titulo_ingles(t)}")
propios = {"Canarias", "Teide", "Lanzarote", "Manrique"}
ejemplos_es = ["HISTORIA NATURAL DE LAS ISLAS CANARIAS", "el Teide y Lanzarote en la obra de manrique"]
for t in ejemplos_es:
    print(f"{t!r:48} → {titulo_espanol(t, propios)}")

assert titulo_ingles("the lord of the rings") == "The Lord of the Rings"
assert titulo_ingles("A TALE OF TWO CITIES") == "A Tale of Two Cities"
assert titulo_espanol(ejemplos_es[0], propios) == "Historia natural de las islas Canarias"
for t in ejemplos_en + ["brave new world", "great expectations"]:
    assert titulo_ingles(titulo_ingles(t)) == titulo_ingles(t)
    if not MENORES & set(t.lower().split()[1:]):
        assert titulo_ingles(t) == t.title()
for t in ejemplos_es:
    assert titulo_espanol(titulo_espanol(t, propios), propios) == titulo_espanol(t, propios)

'the lord of the rings'    → The Lord of the Rings
'A TALE OF TWO CITIES'     → A Tale of Two Cities
'war and peace'            → War and Peace
'of mice and men'          → Of Mice and Men
'HISTORIA NATURAL DE LAS ISLAS CANARIAS'         → Historia natural de las islas Canarias
'el Teide y Lanzarote en la obra de manrique'    → El Teide y Lanzarote en la obra de Manrique
